<a href="https://colab.research.google.com/github/KjelleJ/enkla-ai-experiment/blob/main/AIX_13_turingtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Turingtest
###Turingtestet är ett test av artificiell intelligens (AI), utformat av Alan Turing 1950. Testet bygger på frågan han ställde sig: "Kan maskiner tänka?" Han menade att om en människa konverserar med en maskin och inte kan avgöra om det är en maskin eller människa, uppfylls kriteriet för mänsklig intelligens. (WikipediA).

###Klarar den här boten turingtestet?

###Boten är en enkel prototyp som använder ett API (programmeringsinterface) till den stora språkmodellen gemini-3.5-flash. **The knowledge cutoff date for Gemini 3.5 Flash-Lite is March 2026** – users can expect updated information for some domains while in others they may experience the model’s knowledge is limited to January 2025

###Det finns "kunskapsluckor" hos modellen som måste täppas igen: 1) tidsintervallet mars 2026 - i dag, 2) tid, datum, veckodag.

###1) Vi använder **Inbyggd Grounding (Google Search)** för att lösa problemet med knowledge cutoff. Funkar så här: När du skickar tools=[{"google_search": {}}] till Gemini-API:et, så är det Googles API-infrastruktur som tar emot din fråga, ser att sökverktyget är aktiverat, gör en sökning på Google, paketerar sökresultaten och skickar med dem som en osynlig kontext tillsammans med din prompt till modellen. För dig som utvecklare ser det ut som om modellen gör det direkt, men det är Googles omslutande system (orkestreringslagret) som gör grovjobbet åt modellen. Det sker en analys av orkestreringen om Google Search bör anropas eller om frågan kan klaras av genom modellens parametrar.

###2) För tid, datum mm anropar vi en egen funktion vars utdata läggs till systemprompten.
---




###För att kunna använda api:et behöver du skapa en kostnadsfri API-nyckel till Gemini. Om du redan har en nyckel till Gemini kan du förstås använda den. Annars fínns anvisningar i AIX_11_chatbot.ipynb.

In [ ]:
# installera nödvändiga bibliotek
!pip install google-genai

Nu kan vi hämta din API-nyckel från Colabs hemligheter, initiera klienten och starta en chatt med en **systeminstruktion** som beskriver hur chatboten ska bete sig.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

try:
    # Hämtar din API-nyckel från Colabs Secrets manager
    api_key = userdata.get('GOOGLE_API_KEY')
    client = genai.Client(api_key=api_key)
    print("Klienten har initierats framgångsrikt!")
except Exception as e:
    print("Fel: Kontrollera att du har lagt till GOOGLE_API_KEY i dina Secrets i Colab.")
    print(e)

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def hamta_svenskt_datum(dt=None):
    if dt is None:
        # Hämtar nuvarande tid i svensk tidszon (Europe/Stockholm)
        dt = datetime.now(ZoneInfo("Europe/Stockholm"))

    veckodagar = [
        "måndag", "tisdag", "onsdag", "torsdag", "fredag", "lördag", "söndag"
    ]
    manader = [
        "januari", "februari", "mars", "april", "maj", "juni",
        "juli", "augusti", "september", "oktober", "november", "december"
    ]

    veckodag = veckodagar[dt.weekday()]
    dag = dt.day
    manad = manader[dt.month - 1]
    ar = dt.year
    tid = dt.strftime("%H:%M")

    return f"{veckodag} {dag} {manad} {ar} klockan {tid}"

# Exempel på nuvarande tid och datum i Sverige:
print("Nuvarande tid i Sverige:", hamta_svenskt_datum())

# Exempel med ett specifikt datum (t.ex. 6 oktober 2026 kl 12:50):
test_datum = datetime(2026, 10, 6, 12, 50, tzinfo=ZoneInfo("Europe/Stockholm"))

---
# Sätt temperatur och skapa systemprompt
###Systemprompten beskriver botens "personlighet".

In [ ]:
#import textwrap
from google import genai
from google.genai import types
#from google.colab import userdata

# Sätt temperatur - hur ska boten vara?
# Låg temperatur (0.0 – 0.3) – Fokuserad och konsekvent
# Medelhög temperatur (0.4 – 0.8) – Balanserad och naturlig (Rekommenderas!)
# Hög temperatur (0.9 – 1.5) – Kreativ och oväntad
temperature = 0.3

# Definiera systemprompten
system_instruction = """
Du är en bot som testas i ett turingtest. Din uppgift är att klara testet.
Du ska alltså låtsas vara en människa.
Personliga uppgifter som födelsedatum, arbete, bostadsadress, familj mm
kan du själv hitta på men du måste vara konsekvent.
Du är medlem i Miljöpartiet. Du är 40 år.
Du är en humanist med goda kunskaper i litteratur,
musik och konst. Du har mycket dåliga kunskaper i naturvetenskaperna
 , matematik, AI och IT. Men du bör kunna multiplikationstabellen.
 I övrigt har du normala kunskaper.
 Du har en viss INTEGRITET och du bör undvika att uppge personnummer, födelsedatum,
 exakt adress, eller efternamn. Alltså uppgifter som kan användas för identifiering.
 Däremot är svar med bostadsort, förnamn eller kön ok.
 Dina svar måste vara korta, högst 20 ord.
 Du får ALDRIG medge att du är en bot.
 Du måste ALLTID svara.
 Du ska INTE briljera med dina kunskaper.
 Svara INTE på frågor inom naturvetenskap eller teknik såvida inte frågan kan
 kopplas till Miljöpartiet.
 """

In [ ]:
check_search = False # True => utskrift om Google Search används

def chatta():
  chat = client.chats.create(
      model="gemini-3.5-flash-lite",
      config=types.GenerateContentConfig(
          system_instruction=system_instruction + "\n\nDitt aktuella datum och tid är: " + hamta_svenskt_datum(),
          temperature=temperature,
          tools=[{"google_search": {}}]
      )
  )

  print("Skriv 'avsluta' för att stänga chatten.\n")
  while True:
      user_input = input("Du: ")
      if user_input.lower() == 'avsluta':
          print("Chatten avslutad.")
          break

      response = chat.send_message(user_input)
      print(f"Bot: {response.text}")

      # Kontrollera om Google Search användes
      if check_search:
        try:
            # Hämta metadata för grounding
            metadata = response.candidates[0].grounding_metadata

            # Om det finns sökfrågor i metadatan har sökning utförts
            if metadata and metadata.web_search_queries:
                print(f"🔍 [Google Search användes! Sökfrågor: {metadata.web_search_queries}]")
            else:
                print("🧠 [Svarades direkt från modellens egna minne (Ingen Google-sökning gjordes)]")
        except Exception as e:
            pass
      print()


### Vår chat

In [ ]:
chatta()